# 07 · Barrido de resolución y descongelado

Dos decisiones que hoy están tomadas por criterio y no por medición.

**Resolución.** Los JPEG procesados son de 448×448 y el pipeline los reescala a
320, descartando la mitad de los píxeles. Las cuatro confusiones principales del
docente son pares congenéricos —`heliconius_erato`/`melpomene` entre ellos—, que
se separan por detalle fino del patrón alar. Entrenar a 448 no reescala nada.

**Fracción descongelada.** El 0.30 se eligió sin fase de calentamiento. Ahora que
la cabeza llega entrenada al ajuste, descongelar más es menos arriesgado y el
óptimo pudo moverse.

Se mide sobre el estudiante por costo. Si 448 gana, conviene confirmarlo en
`efficientnet-v2-s` antes de comprometer las trece corridas: la pregunta importa
más en el docente, que es donde están esas confusiones.

Cuatro combinaciones, unos 30 minutos. El arm de 448 cuesta el doble por época,
y ese costo es la mitad de la decisión: duplicar el cómputo de trece notebooks
es la diferencia entre caber y no caber en las 30 horas semanales.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

if Path("/kaggle").exists():
    from kaggle_secrets import UserSecretsClient

    CLON = "/tmp/ButterflyModeling"
    AYUDA = "!f() { echo username=x-access-token; echo password=$GITHUB_TOKEN; }; f"
    os.environ["GITHUB_TOKEN"] = UserSecretsClient().get_secret("GITHUB_TOKEN")
    if not Path(CLON).exists():
        subprocess.run(["git", "-c", f"credential.helper={AYUDA}", "clone",
                        "https://github.com/CCSandoval/ButterflyModeling.git", CLON], check=True)
    sys.path.insert(0, CLON)
    os.chdir(CLON)
    print("clon en", CLON, "·", subprocess.run(
        ["git", "log", "-1", "--format=%h %s"], cwd=CLON,
        capture_output=True, text=True).stdout.strip())

In [ ]:
import os
import sys
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "corpus.json").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print("Raíz del proyecto:", ROOT)

In [ ]:
import time

import pandas as pd
import tensorflow as tf

from scripts import architectures, corpus
from scripts.architectures import buildModel, descongelarCola, preprocessFn
from scripts.dataPipeline import aumentadorFotometrico, loadSplit, preparar

ARQUITECTURA = "mobilenet_v3_small"
RESOLUCIONES = (320, 448)
FRACCIONES = (0.30, 0.60)

LOTE = 32
EPOCAS_CALENTAMIENTO = 5
EPOCAS_AJUSTE = 8
TASA = 1e-4
SEMILLA = 42

corpus.materializar()
preprocess = preprocessFn(ARQUITECTURA)
print(f"{ARQUITECTURA} · {len(RESOLUCIONES) * len(FRACCIONES)} combinaciones")

In [ ]:
filas = []
for lado in RESOLUCIONES:
    architectures.IMG_SIZE = (lado, lado)
    dsTrainCrudo, clases = loadSplit("dataset/train", (lado, lado), LOTE)
    dsValCrudo, _ = loadSplit("dataset/validate", (lado, lado), LOTE, shuffle=False)
    dsT = preparar(dsTrainCrudo, preprocess, aumentadorFotometrico())
    dsV = preparar(dsValCrudo, preprocess)

    for fraccion in FRACCIONES:
        tf.keras.utils.set_random_seed(SEMILLA)
        modelo = buildModel(ARQUITECTURA, len(clases))
        modelo.compile(optimizer=tf.keras.optimizers.Adam(TASA),
                       loss="categorical_crossentropy", metrics=["accuracy"])
        modelo.fit(dsT, validation_data=dsV, epochs=EPOCAS_CALENTAMIENTO, verbose=0)

        descongelarCola(modelo, fraccion)
        modelo.compile(optimizer=tf.keras.optimizers.Adam(TASA),
                       loss="categorical_crossentropy", metrics=["accuracy"])
        inicio = time.time()
        h = modelo.fit(dsT, validation_data=dsV, epochs=EPOCAS_AJUSTE, verbose=0).history
        segundos = (time.time() - inicio) / EPOCAS_AJUSTE

        filas.append({
            "resolucion": lado,
            "descongelado": fraccion,
            "val_acc": h["val_accuracy"][-1],
            "val_loss": h["val_loss"][-1],
            "seg_por_epoca": round(segundos, 1),
        })
        print(f"{lado}px  ft={fraccion:.2f}  val={h['val_accuracy'][-1]:.4f}  "
              f"{segundos:.1f} s/época")
        del modelo

    tf.keras.backend.clear_session()

barrido = pd.DataFrame(filas)
barrido

## Lectura

La referencia es la fila `320 / 0.30`, que es la configuración actual.

Subir la resolución solo se justifica si la ganancia en `val_acc` paga el costo
en `seg_por_epoca`. Subir el descongelado solo cuesta calidad si sale peor, así
que ahí basta con que gane.